# Complex Schema

> **Source:** `repo1/output_parsers_final.py` → `demo_complex_schema()`


## Imports


In [ ]:
from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List, Optional
from dotenv import load_dotenv


## Configuration & Setup


In [ ]:
load_dotenv()

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## Helper Function: `exercise_structured_extraction`


In [ ]:
def exercise_structured_extraction():
    """
    EXERCISE: Create a schema and chain that extracts:
    - Movie title
    - Year released
    - Director
    - Main actors (list)
    - Genre
    - Rating (1-10)

    Test with a movie description.
    """

    class Movie(BaseModel):
        title: str = Field(description="Movie title")
        year: int = Field(description="Year released")
        director: str = Field(description="Director name")
        actors: List[str] = Field(description="Main actors")
        genre: str = Field(description="Primary genre")
        rating: int = Field(description="Rating from 1-10", ge=1, le=10)

    structured_model = model.with_structured_output(Movie)

    prompt = ChatPromptTemplate.from_template(
        "Extract movie information from this review:\n\n{review}"
    )

    chain = prompt | structured_model

    result = chain.invoke(
        {
            "review": "The Dark Knight (2008) directed by Christopher Nolan is an "
            "absolute masterpiece. Christian Bale and Heath Ledger deliver "
            "incredible performances in this action thriller. 10/10!"
        }
    )

    print(f"Title: {result.title}")
    print(f"Year: {result.year}")
    print(f"Director: {result.director}")
    print(f"Actors: {result.actors}")
    print(f"Genre: {result.genre}")
    print(f"Rating: {result.rating}/10")


## Demo: Complex Schema


In [ ]:
def demo_complex_schema():
    """Complex nested schema with structured output."""

    class Address(BaseModel):
        street: str
        city: str
        country: str

    class Company(BaseModel):
        name: str
        industry: str
        employee_count: int
        headquarters: Address
        products: List[str]

    structured_model = model.with_structured_output(Company)

    prompt = ChatPromptTemplate.from_template(
        "Extract company information from: {text}"
    )

    chain = prompt | structured_model

    result = chain.invoke(
        {
            "text": "Apple Inc. is a tech company with 160,000 employees based in "
            "Cupertino, California, USA. They make iPhones, MacBooks, and iPads."
        }
    )

    print(f"Company: {result.name}")
    print(f"Industry: {result.industry}")
    print(f"Employees: {result.employee_count}")
    print(f"HQ: {result.headquarters.city}, {result.headquarters.country}")
    print(f"Products: {result.products}")


## Execute


In [ ]:
demo_complex_schema()
